# Setup: EvalHub Service Deployment & SDK Configuration

This notebook deploys the [EvalHub](https://github.com/eval-hub/eval-hub) service on OpenShift and configures the [eval-hub-sdk](https://github.com/eval-hub/eval-hub-sdk) to run LLM evaluations (including [lm-evaluation-harness](https://github.com/EleutherAI/lm-evaluation-harness)) through a centralized REST API with **MLflow experiment tracking**.

## What is EvalHub?

EvalHub is a lightweight REST API service that orchestrates LLM evaluations across multiple backends. It:

- Routes evaluation requests to frameworks like **lm-evaluation-harness**, RAGAS, GuideLLM, LightEval, and more
- Tracks experiments via **MLflow** (metrics, parameters, artifacts)
- Runs natively on **OpenShift** via the TrustyAI Operator
- Supports a **"Bring Your Own Framework" (BYOF)** approach through the SDK

## EvalHub vs. LMEvalJob (1_LMEval_setup.ipynb)

| Feature | LMEvalJob (direct) | EvalHub (Phase 2) |
|---------|--------------------|---------|
| Interface | Kubernetes CR (YAML) | REST API + Python SDK |
| Frameworks | lm-evaluation-harness only | Multiple (lm-eval, RAGAS, LightEval, ...) |
| Experiment tracking | Manual | Built-in MLflow integration |
| Multi-benchmark jobs | One task per CR | Multiple benchmarks per request |
| Result management | Pod logs / CR status | Centralized API + MLflow UI |

## Prerequisites

- Completed **0_model_deploy.ipynb** (model deployed on OpenShift AI)
- Completed **1_LMEval_setup.ipynb** (RBAC and secrets configured)
- TrustyAI Operator installed on the cluster

---

## Part A: Deploy EvalHub Service on OpenShift

Before using the SDK, the EvalHub service and MLflow must be running on the cluster. This section walks through the deployment.

### Step 0: First-Time Setup (Edit & Run Once)

After cloning, **edit the values in the cell below**, then run it.
This creates the `.env` file and installs dependencies for all notebooks.

| Scenario | What to set |
|----------|-------------|
| **Cluster owner** (deploying EvalHub yourself) | `NAMESPACE`, `MODEL_NAME`, `HF_TOKEN`, `HF_MODEL_ID` |
| **Workshop participant** (using someone else's cluster) | All 6 values from cluster owner: `NAMESPACE`, `MODEL_NAME`, `EVALHUB_URL`, `EVALHUB_AUTH_TOKEN`, `MLFLOW_TRACKING_URI`, `BASE_URL` |

In [ ]:
import os

# ============================================================
#  >>> Edit these values for YOUR environment <<<
# ============================================================
NAMESPACE        = "CHANGE_ME"                  # oc project name (e.g. "my-project")
MODEL_NAME       = "CHANGE_ME"                  # InferenceService name, run: oc get inferenceservice -n <namespace>
HF_TOKEN         = "CHANGE_ME"                  # Hugging Face token (https://huggingface.co/settings/tokens)
HF_MODEL_ID      = "CHANGE_ME"                  # HF model ID (e.g. "google/gemma-4-E2B-it")

# --- Shared cluster (workshop participant): set these from cluster owner ---
EVALHUB_URL        = ""                         # e.g. "https://evalhub-myproject.apps.cluster.example.com"
EVALHUB_AUTH_TOKEN = ""                         # SA token from cluster owner (leave empty if you have oc login)
MLFLOW_TRACKING_URI = ""                        # e.g. "https://mlflow-myproject.apps.cluster.example.com"

# --- Auto-derived (no need to change) ---
BASE_URL         = f"https://{MODEL_NAME}-predictor.{NAMESPACE}.svc.cluster.local:8443/v1"
RUNTIME_NAME     = "vllm-cuda-runtime-gemma4"
HF_TOKEN_SECRET  = "hf-token"
LIMIT            = "5"
BATCH_SIZE       = "8"
if not MLFLOW_TRACKING_URI:
    MLFLOW_TRACKING_URI = "https://mlflow.redhat-ods-applications.svc:8443"

# --- Write .env file ---
env_path = os.path.join(os.path.dirname(os.getcwd()), ".env")
with open(env_path, "w") as f:
    f.write(f"""NAMESPACE={NAMESPACE}
MODEL_NAME={MODEL_NAME}
BASE_URL={BASE_URL}
RUNTIME_NAME={RUNTIME_NAME}
HF_MODEL_ID={HF_MODEL_ID}
HF_TOKEN={HF_TOKEN}
HF_TOKEN_SECRET={HF_TOKEN_SECRET}
LIMIT={LIMIT}
BATCH_SIZE={BATCH_SIZE}
EVALHUB_URL={EVALHUB_URL}
EVALHUB_AUTH_TOKEN={EVALHUB_AUTH_TOKEN}
MLFLOW_TRACKING_URI={MLFLOW_TRACKING_URI}
""")
print(f"Created {env_path}")

In [ ]:
%pip install -q -r ../requirements.txt

### Step A-1: Configuration

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv(dotenv_path="../.env", override=True)

NAMESPACE = os.getenv("NAMESPACE", "hyo-project")

print(f"Namespace: {NAMESPACE}")

### Step A-2: Verify TrustyAI Operator

The TrustyAI Operator manages the `EvalHub` Custom Resource. Verify it is installed on the cluster:

In [ ]:
!oc get csv --all-namespaces 2>/dev/null | grep -i trustyai || \
    echo "TrustyAI Operator not found. Install it from OperatorHub first."

### Step A-3: Deploy MLflow via Operator

EvalHub requires an MLflow tracking server. The MLflow Operator (installed with OpenShift AI) manages the `MLflow` CR.
The operator deploys MLflow into `redhat-ods-applications` and automatically creates an HTTPRoute
so the UI is accessible at the Dashboard gateway (`/mlflow`).

In [ ]:
import subprocess, json

def _mlflow_exists(namespace: str) -> dict | None:
    """Check if an MLflow CR already exists in the given namespace."""
    r = subprocess.run(
        ["oc", "get", "mlflow", "mlflow", "-n", namespace, "-o", "json"],
        capture_output=True, text=True,
    )
    if r.returncode == 0:
        return json.loads(r.stdout)
    return None

existing = _mlflow_exists(NAMESPACE)
if existing:
    status = existing.get("status", {})
    url = status.get("url", "N/A")
    addr = status.get("address", {}).get("url", "N/A")
    ready = any(
        c.get("type") == "Available" and c.get("status") == "True"
        for c in status.get("conditions", [])
    )
    print(f"MLflow CR already exists in '{NAMESPACE}' — skipping creation.")
    print(f"  Internal: {addr}")
    print(f"  Portal:   {url}")
    print(f"  Ready:    {ready}")
else:
    mlflow_cr = f"""apiVersion: mlflow.opendatahub.io/v1
kind: MLflow
metadata:
  name: mlflow
  namespace: {NAMESPACE}
spec:
  replicas: 1
  backendStoreUri: "sqlite:////mlflow/mlflow.db"
  serveArtifacts: true
  artifactsDestination: "file:///mlflow/artifacts"
  storage:
    accessModes:
      - ReadWriteOnce
    resources:
      requests:
        storage: 5Gi
"""
    with open("/tmp/mlflow-cr.yaml", "w") as f:
        f.write(mlflow_cr)
    print("MLflow CR YAML generated:")
    print(mlflow_cr)

In [ ]:
import time

if existing:
    print("MLflow already deployed — nothing to do.")
else:
    !oc apply -f /tmp/mlflow-cr.yaml
    print("Waiting for MLflow to become ready...")
    for i in range(24):
        time.sleep(5)
        info = _mlflow_exists(NAMESPACE)
        if info:
            conds = info.get("status", {}).get("conditions", [])
            if any(c.get("type") == "Available" and c.get("status") == "True" for c in conds):
                url = info["status"].get("url", "")
                addr = info["status"].get("address", {}).get("url", "")
                print(f"\nMLflow is ready!")
                print(f"  Internal: {addr}")
                print(f"  Portal:   {url}")
                break
        print(f"  waiting... ({(i+1)*5}s)")
    else:
        print("WARNING: MLflow did not become ready within 120s. Check: oc get mlflow -n", NAMESPACE)

MLFLOW_SVC_URL = (_mlflow_exists(NAMESPACE) or {}).get("status", {}).get("address", {}).get("url", "")
if not MLFLOW_SVC_URL:
    MLFLOW_SVC_URL = f"http://mlflow.{NAMESPACE}.svc.cluster.local:5000"
    print(f"(fallback) MLFLOW_SVC_URL = {MLFLOW_SVC_URL}")
else:
    print(f"MLFLOW_SVC_URL = {MLFLOW_SVC_URL}")

### Step A-4: Deploy EvalHub via the TrustyAI Operator

Create an `EvalHub` Custom Resource. The TrustyAI Operator will reconcile it into a running EvalHub service with the configured MLflow connection.

Key fields:
- `MLFLOW_TRACKING_URI` — Points to your MLflow service
- `replicas` — Number of EvalHub instances

In [ ]:
evalhub_cr_yaml = f"""apiVersion: trustyai.opendatahub.io/v1alpha1
kind: EvalHub
metadata:
  name: evalhub
  namespace: {NAMESPACE}
spec:
  replicas: 1
  providers:
    - garak
    - garak-kfp
    - lm-evaluation-harness
    - guidellm
  database:
    type: sqlite
  env:
    - name: MLFLOW_TRACKING_URI
      value: "{MLFLOW_SVC_URL}"
"""

with open("/tmp/evalhub-cr.yaml", "w") as f:
    f.write(evalhub_cr_yaml)

print("EvalHub CR YAML:")
print(evalhub_cr_yaml)
print(f"(MLFLOW_TRACKING_URI = {MLFLOW_SVC_URL})")

In [ ]:
!oc apply -f /tmp/evalhub-cr.yaml

In [ ]:
!oc describe evalhub evalhub -n {NAMESPACE}
print()
!oc get pods -n {NAMESPACE} | grep evalhub
print()
!oc logs -l app=evalhub -n {NAMESPACE} --tail=20

### Step A-5: Verify EvalHub Deployment

Wait for the EvalHub pod to become ready and check its status:

In [ ]:
!oc get evalhub -n {NAMESPACE}
print()
!oc get pods -n {NAMESPACE} | grep -E "evalhub|mlflow"

### Step A-6: Get the EvalHub Service URL

The URL depends on **where this notebook is running**:

| Running From | EVALHUB_URL | Setup |
|---|---|---|
| **OpenShift Workbench** (cluster internal) | `https://evalhub.<ns>.svc.cluster.local:8443` | no confiuration |
| **Local PC** (cluster external) | `https://localhost:8443` | `oc port-forward` needed |

In [ ]:
import subprocess, socket, time, httpx

result = subprocess.run(
    ["oc", "get", "svc", "evalhub", "-n", NAMESPACE,
     "-o", "jsonpath={.metadata.name}"],
    capture_output=True, text=True,
)
svc_name = result.stdout.strip() or "evalhub"
EVALHUB_CLUSTER_URL = f"https://{svc_name}.{NAMESPACE}.svc.cluster.local:8443"
EVALHUB_LOCAL_URL = "https://localhost:8443"

def _is_port_open(port: int = 8443) -> bool:
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        s.settimeout(1)
        return s.connect_ex(("127.0.0.1", port)) == 0

def _start_port_forward(namespace: str, port: int = 8443, retries: int = 6):
    """Start oc port-forward in the background if not already running."""
    if _is_port_open(port):
        print(f"Port {port} already in use — port-forward likely running.")
        return True
    print(f"Starting: oc port-forward svc/evalhub {port}:{port} -n {namespace}")
    proc = subprocess.Popen(
        ["oc", "port-forward", f"svc/evalhub", f"{port}:{port}", "-n", namespace],
        stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
    )
    for i in range(retries):
        time.sleep(3)
        if _is_port_open(port):
            print(f"Port-forward started (pid={proc.pid}).")
            return True
        print(f"  waiting... ({(i+1)*3}s)")
    print("WARNING: port-forward did not become ready.")
    return False

_in_cluster = False
try:
    httpx.get(f"{EVALHUB_CLUSTER_URL}/api/v1/health", verify=False, timeout=3)
    _in_cluster = True
except Exception:
    pass

print("EvalHub Service URLs")
print("=" * 60)
print(f"  Cluster-internal: {EVALHUB_CLUSTER_URL}")
print(f"  Local (port-fwd): {EVALHUB_LOCAL_URL}")
print()

if _in_cluster:
    print("Detected: running INSIDE the cluster — using cluster-internal URL.")
    EVALHUB_URL = EVALHUB_CLUSTER_URL
else:
    print("Detected: running OUTSIDE the cluster — local port-forward required.")
    _start_port_forward(NAMESPACE)
    EVALHUB_URL = EVALHUB_LOCAL_URL

    try:
        resp = httpx.get(f"{EVALHUB_URL}/api/v1/health", verify=False, timeout=5)
        info = resp.json()
        print(f"\n[OK] EvalHub reachable — status={info.get('status')}, version={info.get('build')}")
    except Exception as e:
        print(f"\n[FAIL] EvalHub NOT reachable at {EVALHUB_URL}")
        print(f"       Error: {e}")
        print(f"       Run manually: oc port-forward svc/evalhub 8443:8443 -n {NAMESPACE}")

### Step A-6b: Get the Authentication Token

EvalHub uses OpenShift's authentication system. All API calls require a **Bearer token** in the `Authorization` header. Sources:

1. **`.env` token** (`EVALHUB_AUTH_TOKEN`) — pre-shared SA token for workshop participants or long-lived access
2. **`oc whoami -t`** — current user's session token (auto-detected if `.env` token is empty)

> **Note:** If the token expires you will get `401 Unauthorized`. Re-run `oc login` to refresh, or use a long-lived SA token (see Step A-7).

In [ ]:
import os, subprocess

EVALHUB_AUTH_TOKEN = os.getenv("EVALHUB_AUTH_TOKEN", "")
if EVALHUB_AUTH_TOKEN:
    print(f"Using token from .env: {EVALHUB_AUTH_TOKEN[:10]}...")
else:
    result = subprocess.run(["oc", "whoami", "-t"], capture_output=True, text=True)
    if result.returncode == 0:
        EVALHUB_AUTH_TOKEN = result.stdout.strip()
        print(f"Token from oc whoami: {EVALHUB_AUTH_TOKEN[:10]}...")
    else:
        print("No token available. Set EVALHUB_AUTH_TOKEN in .env or run: oc login <cluster-url>")
        EVALHUB_AUTH_TOKEN = None

### Step A-7: Share Cluster Access (Cluster Owner Only)

If you want **workshop participants without their own cluster** to run evaluations against your EvalHub, run the cell below. It will:

1. **Verify** EvalHub is healthy
2. Create OpenShift **Routes** for EvalHub and MLflow (external HTTPS access)
3. Create a **ServiceAccount** with full EvalHub RBAC and a 72-hour token
4. Create a **testuser** OpenShift user (read-only access to `demo` namespace + MLflow UI)
5. **Register** the Korean MCQ adapter provider (so participants don't need `oc` access)
6. Print all connection details — participants paste these into **Step 0**

> **Skip this step** if all users have direct cluster access (Workbench or `oc login`).

In [12]:
import subprocess

def _ensure_route(name, service, port, namespace):
    """Create a passthrough Route if it doesn't exist, return the external URL."""
    r = subprocess.run(
        ["oc", "get", "route", name, "-n", namespace, "-o", "jsonpath={.spec.host}"],
        capture_output=True, text=True
    )
    if r.returncode == 0 and r.stdout.strip():
        url = f"https://{r.stdout.strip()}"
        print(f"  Route '{name}' exists: {url}")
        return url
    subprocess.run(
        ["oc", "create", "route", "passthrough", name,
         f"--service={service}", f"--port={port}", "-n", namespace],
        check=True
    )
    r = subprocess.run(
        ["oc", "get", "route", name, "-n", namespace, "-o", "jsonpath={.spec.host}"],
        capture_output=True, text=True, check=True
    )
    url = f"https://{r.stdout.strip()}"
    print(f"  Route '{name}' created: {url}")
    return url

# 1. Create Routes
print("Creating Routes...")
EVALHUB_EXTERNAL_URL = _ensure_route("evalhub", "evalhub", "8443", NAMESPACE)

MLFLOW_NS = "redhat-ods-applications"
MLFLOW_EXTERNAL_URL = _ensure_route("mlflow", "mlflow", "8443", MLFLOW_NS)

# 2. Create ServiceAccount + token (72h)
SA_NAME = "evalhub-workshop"
subprocess.run(["oc", "create", "sa", SA_NAME, "-n", NAMESPACE],
               capture_output=True, text=True)

# 3. Grant RBAC so the SA token can fully access EvalHub API
cr_yaml = f"""\
apiVersion: rbac.authorization.k8s.io/v1
kind: ClusterRole
metadata:
  name: evalhub-workshop-full-access
rules:
- apiGroups: ["trustyai.opendatahub.io"]
  resources: ["evaluations", "providers", "benchmarks", "jobs", "collections", "experiments"]
  verbs: ["get", "list", "create", "update", "patch", "delete"]
- apiGroups: ["trustyai.opendatahub.io"]
  resources: ["evalhubs", "evalhubs/proxy"]
  verbs: ["get", "create"]
- apiGroups: ["batch"]
  resources: ["jobs"]
  verbs: ["get", "list", "create", "delete"]
- apiGroups: [""]
  resources: ["configmaps"]
  verbs: ["get", "list", "create", "update", "delete"]
- apiGroups: ["mlflow.kubeflow.org"]
  resources: ["experiments"]
  verbs: ["get", "list", "create", "update"]
"""
with open("/tmp/evalhub-workshop-role.yaml", "w") as f:
    f.write(cr_yaml)
subprocess.run(["oc", "apply", "-f", "/tmp/evalhub-workshop-role.yaml"], capture_output=True, text=True)
subprocess.run(
    ["oc", "create", "clusterrolebinding", f"{SA_NAME}-full",
     "--clusterrole=evalhub-workshop-full-access",
     f"--serviceaccount={NAMESPACE}:{SA_NAME}"],
    capture_output=True, text=True,
)
print("  ServiceAccount RBAC configured")

# 4. Create testuser for MLflow UI access (htpasswd)
WORKSHOP_USER = "testuser"
WORKSHOP_PASS = "openshift123"

_htpasswd_secret = "htpasswd-secret"
_idp_name = subprocess.run(
    ["oc", "get", "oauth", "cluster", "-o", "jsonpath={.spec.identityProviders[0].htpasswd.fileData.name}"],
    capture_output=True, text=True
).stdout.strip()
if _idp_name:
    _htpasswd_secret = _idp_name

_extract = subprocess.run(
    ["oc", "get", "secret", _htpasswd_secret, "-n", "openshift-config",
     "-o", "jsonpath={.data.htpasswd}"],
    capture_output=True, text=True
)
if _extract.returncode == 0:
    import base64, tempfile
    _htpasswd_path = "/tmp/htpasswd-workshop"
    with open(_htpasswd_path, "wb") as f:
        f.write(base64.b64decode(_extract.stdout))

    _existing_users = open(_htpasswd_path).read()
    if WORKSHOP_USER not in _existing_users:
        subprocess.run(["htpasswd", "-bB", _htpasswd_path, WORKSHOP_USER, WORKSHOP_PASS],
                       capture_output=True, text=True)
        subprocess.run(
            ["oc", "set", "data", f"secret/{_htpasswd_secret}", "-n", "openshift-config",
             f"--from-file=htpasswd={_htpasswd_path}"],
            capture_output=True, text=True
        )
        print(f"  OpenShift user '{WORKSHOP_USER}' created (password: {WORKSHOP_PASS})")
    else:
        print(f"  OpenShift user '{WORKSHOP_USER}' already exists")

    # Read-only RBAC: demo namespace view + EvalHub read + MLflow read
    _ro_yaml = """\
apiVersion: rbac.authorization.k8s.io/v1
kind: ClusterRole
metadata:
  name: evalhub-readonly
rules:
- apiGroups: ["trustyai.opendatahub.io"]
  resources: ["evaluations", "providers", "benchmarks", "jobs", "collections", "experiments"]
  verbs: ["get", "list"]
- apiGroups: ["trustyai.opendatahub.io"]
  resources: ["evalhubs", "evalhubs/proxy"]
  verbs: ["get"]
"""
    with open("/tmp/evalhub-readonly.yaml", "w") as f:
        f.write(_ro_yaml)
    subprocess.run(["oc", "apply", "-f", "/tmp/evalhub-readonly.yaml"], capture_output=True, text=True)
    subprocess.run(["oc", "adm", "policy", "add-role-to-user", "view", WORKSHOP_USER, "-n", NAMESPACE],
                   capture_output=True, text=True)
    subprocess.run(["oc", "adm", "policy", "add-cluster-role-to-user", "evalhub-readonly", WORKSHOP_USER],
                   capture_output=True, text=True)
    subprocess.run(["oc", "adm", "policy", "add-cluster-role-to-user",
                    "trustyai-service-operator-evalhub-mlflow-access", WORKSHOP_USER],
                   capture_output=True, text=True)
    print(f"  RBAC configured for '{WORKSHOP_USER}'")
else:
    print("  WARNING: Could not extract htpasswd secret — skip testuser creation")

# 5. Generate SA token (72h)
token_result = subprocess.run(
    ["oc", "create", "token", SA_NAME, "--duration=72h", "-n", NAMESPACE],
    capture_output=True, text=True, check=True
)
WORKSHOP_TOKEN = token_result.stdout.strip()
print(f"  Token generated (expires in 72h)")

# 5. Verify EvalHub health
import httpx
try:
    resp = httpx.get(f"{EVALHUB_EXTERNAL_URL}/api/v1/health", verify=False, timeout=5)
    health = resp.json()
    print(f"  EvalHub health: {health['status']} (build {health.get('build', '?')})")
except Exception as e:
    print(f"  WARNING: EvalHub health check failed: {e}")

# 6. Register Korean MCQ provider (idempotent)
import yaml, pathlib
KOREAN_PROVIDER_YAML = pathlib.Path("../adapters/korean-mcq/provider.yaml")
_admin_token = subprocess.run(["oc", "whoami", "-t"], capture_output=True, text=True).stdout.strip()

_existing = httpx.get(
    f"{EVALHUB_EXTERNAL_URL}/api/v1/evaluations/providers",
    headers={"Authorization": f"Bearer {_admin_token}", "X-Tenant": NAMESPACE},
    verify=False, timeout=10,
).json()
_korean_id = None
for p in _existing.get("items", []):
    if p.get("name") == "Korean MCQ Evaluation":
        _korean_id = p["resource"]["id"]
        break

if _korean_id:
    print(f"  Korean MCQ provider already registered (id={_korean_id})")
elif KOREAN_PROVIDER_YAML.exists():
    raw = KOREAN_PROVIDER_YAML.read_text().replace("${NAMESPACE}", NAMESPACE)
    provider_def = yaml.safe_load(raw)
    resp = httpx.post(
        f"{EVALHUB_EXTERNAL_URL}/api/v1/evaluations/providers",
        headers={"Authorization": f"Bearer {_admin_token}", "Content-Type": "application/json", "X-Tenant": NAMESPACE},
        json=provider_def, verify=False, timeout=10,
    )
    if resp.status_code in (200, 201):
        _korean_id = resp.json()["resource"]["id"]
        print(f"  Korean MCQ provider registered (id={_korean_id})")
    else:
        print(f"  WARNING: Korean MCQ registration failed ({resp.status_code}): {resp.text[:200]}")
else:
    print(f"  WARNING: {KOREAN_PROVIDER_YAML} not found — skip provider registration")

print("\n" + "=" * 60)
print("Share these values with workshop participants (Step 0):")
print("=" * 60)
print(f'EVALHUB_URL         = "{EVALHUB_EXTERNAL_URL}"')
print(f'EVALHUB_AUTH_TOKEN  = "{WORKSHOP_TOKEN[:20]}..."')
print(f'MLFLOW_TRACKING_URI = "{MLFLOW_EXTERNAL_URL}"')
print(f'NAMESPACE           = "{NAMESPACE}"')
print(f'MODEL_NAME          = "{MODEL_NAME}"')
print(f'BASE_URL            = "{BASE_URL}"')
print("=" * 60)
print(f"\nMLflow UI login (for viewing evaluation results):")
print(f"  Username: {WORKSHOP_USER}")
print(f"  Password: {WORKSHOP_PASS}")
# MLflow portal URL (served via Dashboard gateway with OAuth)
_mlflow_portal = subprocess.run(
    ["oc", "get", "mlflow", "mlflow", "-o", "jsonpath={.status.url}"],
    capture_output=True, text=True
).stdout.strip() or f"{MLFLOW_EXTERNAL_URL}/mlflow/"

print(f"\nFull token (copy this):\n{WORKSHOP_TOKEN}")
print(f"\nMLflow UI (open in browser — requires OpenShift login):")
print(f"  {_mlflow_portal}")
print(f"\nMLflow API (for notebook programmatic access):")
print(f"  {MLFLOW_EXTERNAL_URL}")

Creating Routes...
  Route 'evalhub' exists: https://evalhub-demo.apps.openshift-cluster.sandbox3031.opentlc.com
  Route 'mlflow' exists: https://mlflow-redhat-ods-applications.apps.openshift-cluster.sandbox3031.opentlc.com
  ServiceAccount RBAC configured

Share these values with workshop participants (Step 0):
EVALHUB_URL         = "https://evalhub-demo.apps.openshift-cluster.sandbox3031.opentlc.com"
EVALHUB_AUTH_TOKEN  = "eyJhbGciOiJSUzI1NiIs..."
MLFLOW_TRACKING_URI = "https://mlflow-redhat-ods-applications.apps.openshift-cluster.sandbox3031.opentlc.com"
NAMESPACE           = "demo"
MODEL_NAME          = "gemma4-e2b-deployment"
BASE_URL            = "https://gemma4-e2b-deployment-predictor.demo.svc.cluster.local:8443/v1"

Full token (copy this):
eyJhbGciOiJSUzI1NiIsImtpZCI6IlhsR19WOHl6YVhIN1FSRXRNSjhYbUFkNFR5MFIwQ1pzMFRNSkpTamhwTDgifQ.eyJhdWQiOlsiaHR0cHM6Ly9rdWJlcm5ldGVzLmRlZmF1bHQuc3ZjIl0sImV4cCI6MTc4NjgxMDM3NSwiaWF0IjoxNzg2NTUxMTc1LCJpc3MiOiJodHRwczovL2t1YmVybmV0ZXMuZGVmYXVsdC5zdm

### Step A-8: Health Check

Verify the EvalHub service is responding before proceeding to SDK setup:

In [ ]:
import httpx

_urls = [EVALHUB_CLUSTER_URL, EVALHUB_LOCAL_URL] if _in_cluster else [EVALHUB_LOCAL_URL, EVALHUB_CLUSTER_URL]

for url in _urls:
    try:
        resp = httpx.get(f"{url}/api/v1/health", verify=False, timeout=3)
        print(f"[OK] {url}")
        print(f"     {resp.json()}")
        break
    except Exception:
        print(f"[--] {url} (unreachable)")
else:
    print("\nEvalHub is not reachable.")
    print("Attempting to (re)start port-forward...")
    if _start_port_forward(NAMESPACE):
        try:
            resp = httpx.get(f"{EVALHUB_LOCAL_URL}/api/v1/health", verify=False, timeout=3)
            print(f"\n[OK] {EVALHUB_LOCAL_URL}")
            print(f"     {resp.json()}")
        except Exception:
            print(f"[FAIL] Still unreachable after port-forward. Check 'oc' login and cluster status.")

---

## Part B: Configure the EvalHub SDK

Now that the EvalHub service is running, install and configure the Python SDK.

### Step B-1: Install the EvalHub SDK

The `eval-hub-sdk` package provides both a Python client for submitting evaluations and the adapter SDK for building custom frameworks.

In [ ]:
%pip install -q -r ../requirements.txt

### Step B-2: Load Configuration

Configuration is loaded from `../.env`. Update these EvalHub-specific variables with the values discovered in Part A:

| Variable | Description | Example |
|----------|-------------|---------|
| `EVALHUB_URL` | EvalHub service endpoint (Step A-6) | `https://evalhub.my-ns.svc.cluster.local:8443` |
| `MLFLOW_TRACKING_URI` | MLflow server URL (Step A-3) | `http://mlflow.my-ns.svc.cluster.local:5000` |

> **Auth Token:** loaded from `EVALHUB_AUTH_TOKEN` in `.env`. Falls back to `oc whoami -t` if empty.

In [ ]:
import os
import subprocess
import httpx
from dotenv import load_dotenv

load_dotenv(dotenv_path="../.env", override=True)

import sys; sys.path.insert(0, '..')
from utils.port_forward import resolve_evalhub_url

NAMESPACE = os.getenv("NAMESPACE", "hyo-project")
MODEL_NAME = os.getenv("MODEL_NAME", "vllm-gemma4-e2b")
BASE_URL = os.getenv("BASE_URL", f"https://{MODEL_NAME}-predictor.{NAMESPACE}.svc.cluster.local:8443/v1")
EVALHUB_AUTH_TOKEN = os.getenv("EVALHUB_AUTH_TOKEN", "")
if not EVALHUB_AUTH_TOKEN:
    _r = subprocess.run(["oc", "whoami", "-t"], capture_output=True, text=True)
    EVALHUB_AUTH_TOKEN = _r.stdout.strip() if _r.returncode == 0 else None
MLFLOW_TRACKING_URI = os.getenv("MLFLOW_TRACKING_URI", "http://mlflow:5000")
EVALHUB_URL = resolve_evalhub_url(namespace=NAMESPACE)

print(f"Namespace:          {NAMESPACE}")
print(f"Model Name:         {MODEL_NAME}")
print(f"Model Endpoint:     {BASE_URL}")
print(f"EvalHub URL:        {EVALHUB_URL}")
print(f"Auth Token:         {'***' if EVALHUB_AUTH_TOKEN else 'None (no auth)'}")
print(f"MLflow Tracking:    {MLFLOW_TRACKING_URI}")

### Step B-3: Verify EvalHub Connectivity

Check that the SDK can connect to the EvalHub service.

In [ ]:
from evalhub import SyncEvalHubClient

client = SyncEvalHubClient(
    base_url=EVALHUB_URL,
    auth_token=EVALHUB_AUTH_TOKEN,
    insecure=True,
    tenant=NAMESPACE,
)

print(f"EvalHub client initialized: {EVALHUB_URL}")
print(f"Tenant (namespace):         {NAMESPACE}")

### Step B-4: Explore Available Providers and Benchmarks

EvalHub ships with pre-configured providers. Let's list them and their benchmarks.

In [ ]:
try:
    providers = client.providers.list()
    print(f"Available Providers ({len(providers)}):")
    print("=" * 60)
    for provider in providers:
        print(f"\n  Provider: {provider.name}")
        print(f"  ID:       {provider.resource.id}")
        print(f"  Desc:     {provider.description}")
        print(f"  Benchmarks: {len(provider.benchmarks)}")
except Exception as e:
    print(f"Failed to list providers: {e}")

In [ ]:
try:
    benchmarks = client.benchmarks.list()
    print(f"\nAvailable Benchmarks ({len(benchmarks)}):")
    print("=" * 60)
    for bm in benchmarks[:20]:
        print(f"  {bm.id:30s}  category={bm.category or 'N/A':15s}  metrics={bm.metrics}")
    if len(benchmarks) > 20:
        print(f"  ... and {len(benchmarks) - 20} more")
except Exception as e:
    print(f"Failed to list benchmarks: {e}")
    benchmarks = None

### Register Korean MCQ Benchmarks

The default EvalHub catalog does not include Korean-language MCQ benchmarks with **per-question accuracy tracking**.
We register a **custom adapter provider** (`korean_mcq`) that uses a dedicated container image
to evaluate Korean LLMs on CLIcK, HAE-RAE, KMMLU, and KMMLU-HARD benchmarks.

Unlike the built-in `lm-evaluation-harness` provider, this adapter:
- Calls the vLLM API directly with MCQ-formatted prompts
- **Async parallel processing** (configurable concurrency, default 20) for ~10x faster evaluation
- Records per-question answers (correct/incorrect) in CSV
- Reports category-level and supercategory-level accuracy to MLflow
- Generates a detailed markdown report as an OCI artifact
- Robust error handling with retries for rate limits, timeouts, and connection errors

The provider definition lives in [`adapters/korean-mcq/provider.yaml`](../adapters/korean-mcq/provider.yaml)
following the [eval-hub-contrib](https://github.com/eval-hub/eval-hub-contrib) `provider.yaml` format.
See [`adapters/korean-mcq/README.md`](../adapters/korean-mcq/README.md) for full documentation.

#### Build the Korean MCQ Adapter Image

The adapter runs as a container in the cluster. We build the image using OpenShift's internal registry
so that EvalHub can pull it. The image is built in the current `NAMESPACE`.

> **Note:** This only needs to run once per namespace. If the image already exists, the cell will skip the build.

In [ ]:
import subprocess

ADAPTER_DIR = "../adapters/korean-mcq"
IMAGE_NAME = "korean-mcq-adapter"

def _image_exists(namespace: str, name: str) -> bool:
    r = subprocess.run(
        ["oc", "get", "imagestream", name, "-n", namespace],
        capture_output=True, text=True,
    )
    return r.returncode == 0

if _image_exists(NAMESPACE, IMAGE_NAME):
    print(f"Image '{IMAGE_NAME}' already exists in '{NAMESPACE}' — skipping build.")
else:
    print(f"Building '{IMAGE_NAME}' in namespace '{NAMESPACE}'...")
    print("This takes 1-2 minutes on first run.\n")

    r = subprocess.run(
        ["oc", "new-build", "--strategy=docker", "--binary",
         f"--name={IMAGE_NAME}", "-n", NAMESPACE],
        capture_output=True, text=True,
    )
    if r.returncode != 0 and "already exists" not in r.stderr:
        print(f"new-build failed: {r.stderr}")
    else:
        print("BuildConfig created.")

    r = subprocess.run(
        ["oc", "start-build", IMAGE_NAME,
         f"--from-dir={ADAPTER_DIR}", "--follow", "-n", NAMESPACE],
        text=True,
    )
    if r.returncode == 0:
        print(f"\nImage built successfully: {IMAGE_NAME}:latest in {NAMESPACE}")
    else:
        print(f"\nBuild failed (exit={r.returncode}). Check: oc logs bc/{IMAGE_NAME} -n {NAMESPACE}")

In [ ]:
import yaml, json, pathlib, httpx

KOREAN_PROVIDER_YAML = pathlib.Path("../adapters/korean-mcq/provider.yaml")
KOREAN_PROVIDER_NAME = "Korean MCQ Evaluation"

def _register_korean_provider(client, yaml_path: pathlib.Path) -> str | None:
    """Register the Korean MCQ adapter provider via the REST API (idempotent).
    Returns the provider_id (UUID assigned by EvalHub) or None on failure."""
    for p in client.providers.list():
        if p.name == KOREAN_PROVIDER_NAME:
            print(f"Provider '{KOREAN_PROVIDER_NAME}' already registered (id={p.resource.id}).")
            return p.resource.id

    raw = yaml_path.read_text().replace("${NAMESPACE}", NAMESPACE)
    provider_def = yaml.safe_load(raw)

    resp = httpx.post(
        f"{EVALHUB_URL}/api/v1/evaluations/providers",
        headers={
            "Authorization": f"Bearer {EVALHUB_AUTH_TOKEN}",
            "Content-Type": "application/json",
            "X-Tenant": NAMESPACE,
        },
        json=provider_def,
        verify=False,
        timeout=10,
    )
    if resp.status_code in (200, 201):
        data = resp.json()
        pid = data["resource"]["id"]
        n = len(data.get("benchmarks") or [])
        print(f"Provider '{provider_def['name']}' registered (id={pid}, {n} benchmarks)")
        return pid
    else:
        print(f"Registration failed ({resp.status_code}): {resp.text[:200]}")
        return None

KOREAN_PROVIDER_ID = _register_korean_provider(client, KOREAN_PROVIDER_YAML)

benchmarks = client.benchmarks.list()
korean_keywords = ["kmmlu", "haerae", "click", "korean_mcq"]
korean_benchmarks = [
    bm for bm in benchmarks
    if any(kw in bm.id.lower() for kw in korean_keywords)
]
print(f"\nKorean MCQ Benchmarks Found: {len(korean_benchmarks)}")
print("=" * 60)
for bm in korean_benchmarks:
    print(f"  {bm.id:35s}  {bm.name}")

### Step B-5: Configure the Model Endpoint

The `ModelConfig` specifies which model endpoint EvalHub should target. This points to your deployed vLLM InferenceService.

#### Key Parameters

| Parameter | Description | Example |
|-----------|-------------|---------|
| `url` | OpenAI-compatible endpoint URL | `https://model-predictor.ns.svc:8443/v1` |
| `name` | Model name (as registered in vLLM) | `vllm-gemma4-e2b` |
| `auth.secret_ref` | K8s Secret for model auth (optional) | `lmeval-sa-token` |

In [ ]:
from evalhub import ModelConfig
from evalhub.models.api import ModelAuth

TOKENIZER = os.getenv("TOKENIZER", "google/gemma-2b")
HF_TOKEN_SECRET = os.getenv("HF_TOKEN_SECRET", "hf-token")

model = ModelConfig(
    url=BASE_URL,
    name=MODEL_NAME,
    auth=ModelAuth(secret_ref=HF_TOKEN_SECRET),
)

print("Model Configuration:")
print(f"  URL:       {model.url}")
print(f"  Name:      {model.name}")
print(f"  Auth:      secret_ref={model.auth.secret_ref}")
print(f"  Tokenizer: {TOKENIZER} (passed via benchmark parameters)")

#### (Optional) Model Authentication

If your InferenceService has OAuth enabled (`security.opendatahub.io/enable-auth: "true"`), reference a Kubernetes Secret containing the ServiceAccount token:

In [ ]:
from evalhub.models.api import ModelAuth

model_with_auth = ModelConfig(
    url=BASE_URL,
    name=MODEL_NAME,
    auth=ModelAuth(secret_ref="lmeval-sa-token"),
)

print("Model Configuration (with auth):")
print(f"  URL:        {model_with_auth.url}")
print(f"  Name:       {model_with_auth.name}")
print(f"  Auth:       secret_ref={model_with_auth.auth.secret_ref}")

### Step B-6: Configure MLflow Experiment Tracking

EvalHub integrates with MLflow to automatically track evaluation metrics, parameters, and artifacts. When you include an `ExperimentConfig` in your job submission, EvalHub will:

1. Create (or reuse) an MLflow experiment with the given name
2. Log all benchmark metrics (accuracy, f1, etc.) as MLflow metrics
3. Tag the run with model info, benchmark details, and custom tags
4. Store detailed result artifacts

The MLflow connection was configured in **Step A-4** via `MLFLOW_TRACKING_URI` in the EvalHub CR.

#### What MLflow Records

| MLflow Tab | Recorded? | Description |
|------------|-----------|-------------|
| **Overview** | Yes | Job metadata, parameters (model, benchmark, temperature, concurrency, etc.) |
| **Model Metrics** | Yes | `overall_accuracy`, `category_accuracy.*`, `supercategory_accuracy.*` |
| **Artifacts** | Yes | `detailed_results.csv`, `results.json`, `DETAILED_RESULTS.md` |
| **Trace** | Yes | Each LLM call (prompt/response) is recorded as a structured trace span via `MlflowClient.start_trace()` API, visible in the MLflow UI's Traces tab. The adapter connects directly to the MLflow service (bypassing EvalHub proxy). |
| **System Metrics** | No | Requires `psutil` + `mlflow.enable_system_metrics_logging()` in the Pod |

> **Note:** System Metrics are not recorded by design. The Korean MCQ adapter prioritizes
> lightweight, fast execution within a K8s Pod.

#### ExperimentConfig in Job Submission

You control experiment tracking per-job via the `experiment` field:

In [ ]:
from evalhub import ExperimentConfig, ExperimentTag

experiment = ExperimentConfig(
    name="korean-llm-evaluation",
    tags=[
        ExperimentTag(key="model_family", value="gemma-4"),
        ExperimentTag(key="language", value="korean"),
        ExperimentTag(key="environment", value="dev"),
        ExperimentTag(key="team", value="ai-evaluation"),
    ],
)

print("MLflow Experiment Configuration:")
print(f"  Name:  {experiment.name}")
print(f"  Tags:")
for tag in experiment.tags:
    print(f"    {tag.key}: {tag.value}")

### Step B-7: Submit a Single Benchmark Evaluation

Submit a **CLIcK** (Cultural and Linguistic Intelligence in Korean) benchmark evaluation
using the `korean_mcq` provider. CLIcK has 1,995 questions — we limit to 2,000 for a full run.
The adapter evaluates each question individually and reports per-category accuracy.

In [ ]:
from evalhub import BenchmarkConfig, JobSubmissionRequest

single_job_request = JobSubmissionRequest(
    name="click-evaluation",
    description="CLIcK Korean cultural/linguistic MCQ benchmark",
    tags=["korean", "click", "mcq", "culture"],
    model=model,
    benchmarks=[
        BenchmarkConfig(
            id="click",
            provider_id=KOREAN_PROVIDER_ID,
            parameters={
                "temperature": 0.0,
                "max_tokens": 16,
                "limit": 2000,
            },
        ),
    ],
    experiment=experiment,
)

print("Job Submission Request:")
print(f"  Name:       {single_job_request.name}")
print(f"  Model:      {single_job_request.model.name} @ {single_job_request.model.url}")
print(f"  Provider:   {KOREAN_PROVIDER_ID}")
print(f"  Benchmarks: {[b.id for b in single_job_request.benchmarks]}")
print(f"  Experiment: {single_job_request.experiment.name}")

In [ ]:
try:
    job = client.jobs.submit(single_job_request)
    print(f"Job submitted!")
    print(f"  Job ID:        {job.id}")
    print(f"  State:         {job.state}")
    print(f"  MLflow Exp ID: {job.resource.mlflow_experiment_id or 'pending'}")
except Exception as e:
    job = None
    print(f"Failed to submit job: {e}")

### Step B-8: Monitor Job Progress

Poll the job status until it completes.

In [ ]:
import time
from evalhub import JobStatus

if job is None:
    print("Skipped — no job was submitted in the previous cell.")
else:
    TERMINAL_STATES = {JobStatus.COMPLETED, JobStatus.FAILED, JobStatus.CANCELLED, JobStatus.PARTIALLY_FAILED}

    print(f"Monitoring job {job.id}...")
    print("-" * 60)

    while True:
        status = client.jobs.get(job.id)
        state = status.effective_state

        msg = ""
        if status.status and status.status.message:
            msg = f" - {status.status.message.message}"
        print(f"  [{state.value:>10s}]{msg}")

        if state in TERMINAL_STATES:
            break

        time.sleep(10)

    print("-" * 60)
    print(f"Final state: {state.value}")

### Step B-9: View Results

Retrieve the evaluation results, including MLflow run information.

In [ ]:
if job is None:
    print("Skipped — no job was submitted.")
else:
    completed_job = client.jobs.get(job.id)

    if completed_job.results:
        print("Evaluation Results:")
        print("=" * 60)

        if completed_job.results.mlflow_experiment_url:
            print(f"\n  MLflow Experiment: {completed_job.results.mlflow_experiment_url}")

        for bm_result in completed_job.results.benchmarks:
            print(f"\n  Benchmark: {bm_result.id} (provider: {bm_result.provider_id})")
            if bm_result.mlflow_run_id:
                print(f"  MLflow Run ID: {bm_result.mlflow_run_id}")
            print(f"  Metrics:")
            for metric_name, metric_value in bm_result.metrics.items():
                print(f"    {metric_name}: {metric_value}")
    else:
        print("No results available yet.")

### Step B-10: Multi-Benchmark Evaluation

Submit multiple benchmarks in a single request. EvalHub runs them concurrently and tracks all results under one MLflow experiment.

In [ ]:
multi_job_request = JobSubmissionRequest(
    name="korean-mcq-comprehensive-eval",
    description="Multi-benchmark Korean MCQ evaluation (CLIcK + KMMLU)",
    tags=["korean", "comprehensive", "mcq"],
    model=model,
    benchmarks=[
        BenchmarkConfig(
            id="click",
            provider_id=KOREAN_PROVIDER_ID,
            parameters={"temperature": 0.0, "max_tokens": 16, "limit": 2000},
        ),
        BenchmarkConfig(
            id="kmmlu",
            provider_id=KOREAN_PROVIDER_ID,
            parameters={"temperature": 0.0, "max_tokens": 16, "limit": 2000},
        ),
    ],
    experiment=ExperimentConfig(
        name="korean-mcq-comprehensive-evaluation",
        tags=[
            ExperimentTag(key="evaluation_type", value="comprehensive"),
            ExperimentTag(key="model_family", value="gemma-4"),
            ExperimentTag(key="language", value="korean"),
            ExperimentTag(key="adapter", value="korean-mcq"),
        ],
    ),
)

print("Multi-Benchmark Job Request:")
print(f"  Name:       {multi_job_request.name}")
print(f"  Benchmarks: {[b.id for b in multi_job_request.benchmarks]}")
print(f"  Experiment: {multi_job_request.experiment.name}")

# Uncomment to submit:
# multi_job = client.jobs.submit(multi_job_request)
# print(f"\nJob submitted: {multi_job.id}")

### Step B-11: Use Collections for Standardized Evaluations

Collections group benchmarks into reusable evaluation suites. This is useful for certification or compliance workflows.

In [ ]:
try:
    collections = client.collections.list()
    print(f"Available Collections ({len(collections)}):")
    print("=" * 60)
    for coll in collections:
        print(f"\n  Collection: {coll.name}")
        print(f"  ID:         {coll.resource.id}")
        print(f"  Category:   {coll.category}")
        print(f"  Benchmarks: {len(coll.benchmarks)}")
        for bm_ref in coll.benchmarks[:5]:
            print(f"    - {bm_ref.id} (provider: {bm_ref.provider_id})")
        if len(coll.benchmarks) > 5:
            print(f"    ... and {len(coll.benchmarks) - 5} more")
except Exception as e:
    print(f"Failed to list collections: {e}")

### Step B-12: List and Manage Jobs

Review all submitted evaluation jobs.

In [ ]:
try:
    jobs_list = client.jobs.list()
    print(f"Evaluation Jobs ({len(jobs_list)}):")
    print("=" * 60)
    for j in jobs_list:
        state = j.effective_state.value
        exp_name = j.experiment.name if j.experiment else "N/A"
        bm_ids = [b.id for b in j.benchmarks] if j.benchmarks else []
        print(f"  [{state:>16s}] {j.id[:12]}... | {j.name} | exp={exp_name} | benchmarks={bm_ids}")
except Exception as e:
    print(f"Failed to list jobs: {e}")

## Reference: EvalHub SDK Quick Reference

### Client SDK Imports

```python
from evalhub import (
    SyncEvalHubClient,          # Synchronous client (recommended for notebooks)
    AsyncEvalHubClient,         # Async client (for production apps)
    ModelConfig,                # Model endpoint configuration
    BenchmarkConfig,            # Benchmark selection and parameters
    JobSubmissionRequest,       # Full job request
    ExperimentConfig,           # MLflow experiment settings
    ExperimentTag,              # MLflow tags
    CollectionRef,              # Reference to a benchmark collection
    EvaluationExports,          # OCI artifact export config
    EvaluationExportsOCI,       # OCI-specific export settings
    OCICoordinates,             # OCI registry coordinates
    JobStatus,                  # Job status enum
)
```

### Key API Patterns

```python
# Initialize client
client = SyncEvalHubClient(
    base_url="https://evalhub:8443",
    auth_token="...",           # Optional: SA token or API key
    insecure=True,              # Skip TLS verification (dev only)
    tenant="my-namespace",      # Kubernetes namespace
)

# Explore resources (all return plain lists)
providers: list[Provider]     = client.providers.list()
benchmarks: list[Benchmark]   = client.benchmarks.list()
collections: list[Collection] = client.collections.list()

# Submit a job
job: EvaluationJob = client.jobs.submit(request)

# Monitor and retrieve results
status: EvaluationJob = client.jobs.get(job.id)
```

### MLflow Experiment Structure

When an `ExperimentConfig` is provided:

- **Experiment Name**: `{prefix}_{experiment.name}`
- **Tags**: Direct mapping from `experiment.tags`
- **Run**: One MLflow run per evaluation request
- **Metrics**: Benchmark scores logged automatically
- **Parameters**: Model config and benchmark settings logged
- **Artifacts**: Detailed result files stored

### Useful Links

- [EvalHub GitHub](https://github.com/eval-hub/eval-hub)
- [EvalHub SDK GitHub](https://github.com/eval-hub/eval-hub-sdk)
- [EvalHub API Docs](https://eval-hub.github.io/eval-hub/)
- [MLflow Integration Guide](https://github.com/eval-hub/eval-hub/blob/main/MLFLOW.md)

## Done!

You've now configured the EvalHub SDK and learned how to:

1. **Connect** to the EvalHub service with the Python SDK
2. **Configure a model endpoint** pointing to your deployed InferenceService
3. **Set up MLflow experiment tracking** with tags and experiment names
4. **Submit evaluations** using lm-evaluation-harness benchmarks
5. **Monitor** job progress and **retrieve results**
6. **Run multi-benchmark** evaluations in a single request

### Next Steps

- **1_eval_hub_guidellm_benchmark/** -- Run inference performance benchmarks via GuideLLM
- **2_eval_hub_kmcq_benchmark/1_kmcq_benchmark.ipynb** -- Run Korean MCQ benchmark evaluation
- **2_eval_hub_kmcq_benchmark/2_summarize_results.ipynb** -- Summarize results and generate reports
- **3_eval_hub_unified_benchmark/1_unified_benchmark.ipynb** -- Run unified accuracy + performance evaluation